In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%%javascript
IPython.notebook.kernel.execute('nb_name = "' + IPython.notebook.notebook_name + '"')

In [ ]:
# Set notebook name as env variable to enable Wandb code saving
import os

try:
    nb_name
except NameError:
    nb_name = os.path.basename(globals()['__vsc_ipynb_file__'])
os.environ["WANDB_NOTEBOOK_NAME"] = nb_name
print(nb_name)

In [ ]:
import torch

import os
import glob
import re
import wandb
from omegaconf import OmegaConf

from torchinfo import summary

import lightning.pytorch as pl
from lightning.pytorch import seed_everything
from lightning.pytorch.loggers.logger import DummyLogger
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor

from snpgen.utils import instantiate_from_config, save_config, scale_lr_optimizer_config
from snpgen.training.callbacks.progress import SimpleProgressBar
from snpgen.training.loggers import setup_wandb_logger

OmegaConf.register_new_resolver("eval", eval)

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu") 

In [ ]:
NUM_WORKERS = int(os.environ["SLURM_CPUS_PER_TASK"])
NUM_NODES = int(os.environ["SLURM_NNODES"])
ALLOCATED_GPUS_PER_NODE = int(os.environ["SLURM_GPUS_ON_NODE"])
SLURM_JOBID = os.environ["SLURM_JOB_ID"]

In [ ]:
num_gpus = torch.cuda.device_count()
print(f"{num_gpus} GPU(s) available")
print(f"Using {NUM_WORKERS} workers for the DataLoader")

## User Settings

In [ ]:
# ============================================================
# USER SETTINGS — Edit these before running
# ============================================================

# Model configuration
model_size = 'small'         # Options: 'tiny', 'small', 'medium', 'big'
encoder_type = 'encoder'     # Options: 'encoder'
emb_size = 128               # Options: 32, 64, 128, 256
use_discriminator = True

# Dataset and experiment naming
proj_name = 'trait1'
h5_filename = 'snp_dataset_kb10_r0.5_WHITE'
extra_name = f'{model_size}_white'  # Suffix appended to the run name

# Base directory for checkpoints and data
base_scratch_dir = '/path/to/your/snpgen'

# Whether to use Wandb for logging
use_wandb = True

# Load Config

In [ ]:
assert model_size in ['tiny', 'small', 'medium', 'big'], "Invalid model size"
assert emb_size in [32, 64, 128, 256], "Invalid embedding size"
assert encoder_type in ['encoder',], "Invalid encoder type"

base = ['./configs/vae/base.yaml']
base.append(f'./configs/vae/{encoder_type}/base.yaml')

if model_size == 'tiny':
    base.append(f'./configs/vae/{encoder_type}/tiny_emb{emb_size}.yaml')
elif model_size == 'small':
    base.append(f'./configs/vae/{encoder_type}/small_emb{emb_size}.yaml')
elif model_size == 'medium':
    base.append(f'./configs/vae/{encoder_type}/medium_emb{emb_size}.yaml')
elif model_size == 'big':
    base.append(f'./configs/vae/{encoder_type}/big_emb{emb_size}.yaml')
else:
    raise NotImplementedError

if use_discriminator:
    base.append(f'./configs/vae/base_disc.yaml')

In [ ]:
print(f"Loading config from: {base}")
configs = [OmegaConf.load(cfg) for cfg in base]
cli = OmegaConf.from_dotlist([])
config = OmegaConf.merge(*configs, cli)

config_orig = config.copy() # keep a backup of the original config prior to any change

In [ ]:
encoder_config = OmegaConf.to_container(config.model.params.autoencoder_config.params.encoder_config.params, resolve=True)
decoder_config = OmegaConf.to_container(config.model.params.autoencoder_config.params.decoder_config.params, resolve=True)

if use_discriminator:
    discriminator_config = OmegaConf.to_container(config.model.params.loss_config.params.discriminator_config.params, resolve=True)

vae_model_size = config.model.params.autoencoder_config.model_size

In [ ]:
# Scale LR
scale_lr_optimizer_config(config.model.params.optimizer_config, num_gpus=num_gpus)

In [ ]:
seed = config.get('seed', 42)
seed_everything(seed, workers=True)

# Build Dataset

In [ ]:
data_path = os.path.join(base_scratch_dir, f'data/ukb_{proj_name}/')
h5_path = os.path.join(data_path, h5_filename+'.hdf5')

config_orig['dataset_path'] = h5_path

In [ ]:
print(f"Loading Dataset from: {h5_path}")
raw_dataset = instantiate_from_config(config.data.raw_dataset, file_path=h5_path, seed=seed)

In [ ]:
train_dataset = instantiate_from_config(config.data.dataset, raw_dataset.get_split('train'), block_ids=raw_dataset.get_metadata('full', 'block_id'))
val_dataset = instantiate_from_config(config.data.dataset, raw_dataset.get_split('val'), block_ids=raw_dataset.get_metadata('full', 'block_id'))
test_dataset = instantiate_from_config(config.data.dataset, raw_dataset.get_split('test'), block_ids=raw_dataset.get_metadata('full', 'block_id'))

In [ ]:
batch_size = config.data.batch_size
actual_batch_size = batch_size * num_gpus
val_batch_size = config.data.val_batch_size

train_dataloader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
    persistent_workers=True,
    #sampler=ImbalancedDatasetSampler(train_dataset, strategy='inverse_freq'), # balance dataset on labels (which also implicitly performs shuffling)
)

val_dataloader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=val_batch_size,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
    persistent_workers=True,
)

test_dataloader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=val_batch_size,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
    persistent_workers=True,
)

# Build Models

In [ ]:
if 'seq_len' in config:
    config.seq_len = train_dataset.get_seq_len()

In [ ]:
vae_training_wrapper = instantiate_from_config(config.model)

# torch.compile() is quite broken when used with PyTorch Lightning modules, especially for the logging stuff
#vae_training_wrapper = torch.compile(vae_training_wrapper, mode="reduce-overhead", dynamic=True, fullgraph=True)

In [ ]:
# summary(vae_training_wrapper.autoencoder.encoder, input_size=[(3, 3, train_dataset.get_seq_len())], dtypes=[torch.float32], depth=2)

In [ ]:
# summary(vae_training_wrapper.autoencoder.decoder, input_size=[(3, decoder_config['z_channels'], decoder_config['z_dim'])], dtypes=[torch.float32], depth=2)

# summary(vae_training_wrapper.autoencoder.decoder, input_size=[(3, decoder_config['z_dim'])], dtypes=[torch.float32], depth=2)

In [ ]:
# if use_discriminator:
#     print(summary(vae_training_wrapper.loss.discriminator, input_size=[(3, 3, seq_len)], dtypes=[torch.float32], depth=2))

# Train

In [ ]:
actual_emb_size = decoder_config['z_dim']

run_name = f"{proj_name}_vae{f'_disc' if use_discriminator else ''}\
{f'_{encoder_type}' if encoder_type != 'encoder' else ''}\
_emb{emb_size}{f'_actualEmb{actual_emb_size}' if actual_emb_size != emb_size else ''}\
{f'_{extra_name}' if extra_name != '' else ''}-{SLURM_JOBID}"
base_run_dir = os.path.join(base_scratch_dir, f"checkpoints/{proj_name}")
run_dir = f"{base_run_dir}/{run_name}/"
print('run_dir: ', run_dir)

In [ ]:
mixed_precision = config.training.mixed_precision

if mixed_precision:
    if torch.cuda.torch.cuda.is_bf16_supported(including_emulation=False):
        precision = 'bf16-mixed'
    else:
        precision = '16-mixed'
else:
    precision = '32-true'
    
print(precision)

In [ ]:
enable_progress_bar = True

metric_to_monitor = 'val/metrics/recons/accuracy'
filename = f'epoch={{epoch}}-step={{step}}-val_accuracy_recons={{{metric_to_monitor}:.3f}}'

model_ckpt_cb = ModelCheckpoint(
    dirpath=run_dir,
    monitor=metric_to_monitor,
    mode='max',
    filename=filename,
    auto_insert_metric_name=False
)

lr_monitor_cb = LearningRateMonitor(logging_interval='step')

callbacks = [
    lr_monitor_cb,
    model_ckpt_cb,
]

if enable_progress_bar:
    callbacks.append(SimpleProgressBar())

In [ ]:
resolved_config_dict = OmegaConf.to_container(config, resolve=True)

extra_config = {
    "SLURM_JOBID": SLURM_JOBID, "dataset_path": h5_path,
    "encoder_config": encoder_config, "decoder_config": decoder_config,
    "yaml_config": config_orig, "resolved_yaml_config": resolved_config_dict,
    "batch_size": batch_size, "actual_batch_size": actual_batch_size
}
if use_discriminator:
    extra_config['discriminator_config'] = discriminator_config

if use_wandb:
    wandb_logger = setup_wandb_logger(project="SNPgen", name=run_name, save_code=True, save_dir=base_scratch_dir,
                                    group='VAE', tags=['white_ethnicity', proj_name],
                                    extra_config=extra_config, extra_sync_metric="trainer/samples_seen")

# Setup trainer
trainer = pl.Trainer(
    max_epochs=400,
    accelerator="gpu",
    default_root_dir=run_dir,
    devices=num_gpus, # devices=ALLOCATED_GPUS_PER_NODE
    strategy='auto' if num_gpus == 1 else 'ddp',
    logger=wandb_logger if use_wandb else DummyLogger(),
    log_every_n_steps=1,
    enable_checkpointing=True,
    enable_progress_bar=enable_progress_bar,
    callbacks=callbacks,
    precision=precision,
    #limit_train_batches=50, # only for testing
    #limit_val_batches=5, # only for testing
)

save_config(config_orig, run_dir)
trainer.fit(vae_training_wrapper, train_dataloader, val_dataloaders=val_dataloader)

if use_wandb:
    wandb.finish()